# K-Nearest Neighbors: Metric Space Geometry and Hyperparameter Tuning
Accompanies **Unit 4.1: K-Nearest Neighbors (KNN)**.

KNN makes predictions based on local proximity in metric space:
$$\hat{y}(\mathbf{x}) = \arg\max_c \sum_{i \in N_K(\mathbf{x})} \mathbb{I}(y_i = c)$$

Key Hyperparameters to Tune:
- `n_neighbors` ($K$): The central capacity parameter.
  - Small $K$ ($K=1$): Zero training error, but high variance. The decision boundary snakes around individual noise points.
  - Large $K$ ($K \to N$): Oversmooths boundaries, producing high bias. The model simply predicts the majority class everywhere.
- `weights`: `'uniform'` treats all $K$ neighbors equally; `'distance'` weights neighbors by $1/d$, dampening the influence of distant points.
- `p` (Minkowski metric parameter): $p=1$ (Manhattan distance) vs. $p=2$ (Euclidean distance).

In this notebook, we demonstrate the mathematical necessity of standardization and tune $K$ across the bias-variance curve.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score

X, y = make_classification(n_samples=800, n_features=2, n_informative=2, n_redundant=0, n_clusters_per_class=1, random_state=42)

# Simulate unstandardized feature: scale feature 0 by 1,000x
X_unscaled = X.copy()
X_unscaled[:, 0] = X_unscaled[:, 0] * 1000.0

X_train_u, X_test_u, y_train, y_test = train_test_split(X_unscaled, y, test_size=0.3, random_state=42)

### Demonstration: Why Feature Standardization is Mandatory
In unscaled space, Euclidean distance $(x_{i0} - x_{k0})^2 + (x_{i1} - x_{k1})^2$ is completely dominated by feature 0.

In [ ]:
knn_unscaled = KNeighborsClassifier(n_neighbors=5).fit(X_train_u, y_train)
acc_unscaled = knn_unscaled.score(X_test_u, y_test)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train_u)
X_test_s = scaler.transform(X_test_u)

knn_scaled = KNeighborsClassifier(n_neighbors=5).fit(X_train_s, y_train)
acc_scaled = knn_scaled.score(X_test_s, y_test)

print(f"Accuracy without standardization: {acc_unscaled:.4f}")
print(f"Accuracy with standardization:    {acc_scaled:.4f}")

### Tuning $K$ via 5-Fold Cross-Validation: Tracking the Bias-Variance Curve

In [ ]:
k_range = range(1, 40, 2)
cv_scores = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k, weights='distance')
    scores = cross_val_score(knn, X_train_s, y_train, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())

optimal_k = k_range[np.argmax(cv_scores)]
print(f"Optimal K: {optimal_k} with Mean CV Accuracy: {max(cv_scores):.4f}")

plt.figure(figsize=(8, 4.5))
plt.plot(k_range, cv_scores, 'bo-', linewidth=2)
plt.axvline(x=optimal_k, color='crimson', linestyle='--', label=f'Optimal K = {optimal_k}')
plt.xlabel('Number of Neighbors ($K$)', fontsize=11)
plt.ylabel('5-Fold CV Accuracy', fontsize=11)
plt.title('KNN Capacity Tuning: Bias-Variance Trade-Off', fontsize=12, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.show()